### Model Loading

In [1]:
from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose

import torch

import sys

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')



# Specify the path to model config and checkpoint file
config_file = 'config.py'
checkpoint_file = 'weights.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loads checkpoint by local backend from path: weights.pth
Loads checkpoint by local backend from path: weights.pth


VFNet(
  (data_preprocessor): MyPrePro()
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): ResLayer(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      init_cfg={'type': 'Constant', 'val': 0, 'override': {'name': 'norm2'}}
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1)

In [3]:
image_height = 512
image_width = 512
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model

with torch.no_grad():
    output = model(dummy_input)

for o in output:
    for subo in o:
        print(subo.shape)

torch.Size([1, 1, 64, 64])
torch.Size([1, 1, 32, 32])
torch.Size([1, 1, 16, 16])
torch.Size([1, 1, 8, 8])
torch.Size([1, 1, 4, 4])
torch.Size([1, 4, 64, 64])
torch.Size([1, 4, 32, 32])
torch.Size([1, 4, 16, 16])
torch.Size([1, 4, 8, 8])
torch.Size([1, 4, 4, 4])


In [ ]:
for item in output[0]:
    print(item.shape)

In [ ]:
for item in output[1]:
    print(item.shape)

### torch2ONNX (mode1)

In [ ]:
image_height = 512
image_width = 512
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model

onnx_path = 'model.onnx'

torch.onnx.export(
                model,
                dummy_input,
                onnx_path,
            )
print(f"ONNX model exported to {onnx_path}.")

In [ ]:
!mo --input_model ./model.onnx --output_dir ./ --scale 1 --mean_values [0] --model_name end2end 

#### torch2ONNX (mode2): method that employs mmdeploy 

In [ ]:
# EXPORT TO ONNX USING DEPLOYER IN RUNSCRIPTS

In [ ]:
!mo --input_model ./export/end2end.onnx --output_dir ./export/ --scale 1 --mean_values [0] --compress_to_fp16 --model_name end2endIR

### torch2ONNX (mode3): method that employs from mmdeploy.apis.onnx import export 

In [4]:
from mmdeploy.apis.onnx import export
import torch

image_height = 512
image_width = 512
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model


export(model = model,
           args = dummy_input,
           output_path_prefix = '/Data_large/marine/PythonProjects/MMDET/notebooks/OpenVINO/apisonnx',
           backend = 'default',
           input_metas = None,
           context_info = dict(),
           input_names = None,
           output_names = None,
           opset_version = 11,
           dynamic_axes = None,
           verbose = False,
           keep_initializers_as_inputs = None,
           optimize = False)


09/17 09:15:31 - mmengine - INFO - Export PyTorch model to ONNX: /Data_large/marine/PythonProjects/MMDET/notebooks/OpenVINO/apisonnx.onnx.
============= Diagnostic Run torch.onnx.export version 2.0.0+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [ ]:
!mo -h